In [ ]:
# ==============================================================================
# CELL 1: IMPORTS & CONFIGURATION
# ==============================================================================
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW 
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, matthews_corrcoef, roc_auc_score, 
    average_precision_score, confusion_matrix, roc_curve, f1_score,
    precision_score, recall_score, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from datetime import datetime
from safetensors.torch import save_file

# --- CONFIGURATION ---
class Config:
    # Files (Update paths if necessary)
    POSITIVE_CSV = "datasets/fusion_gene_positive_bp_information_with_class_for_modeling.txt"
    NEGATIVE_CSV = "datasets/fusion_gene_negative_bp_information_with_class_for_modeling.txt"
    TEST_CSV = "datasets/fusion_gene_positive_bp_information_with_class_for_testing.txt"
    
    # Extra FASTA Data
    EXTRA_POS_FASTA = "datasets/blast_validated_chimeras.fasta,datasets/cosmic_high_confidence_sequences.fna,datasets/chimeras_43466.fa"
    NEG_FASTA_CANONICAL = "datasets/false_negative_candidates.fasta"
    NEG_FASTA_SYNTHETIC = "datasets/false_positive_candidates.fasta"
    
    # Model & Training
    OUTPUT_DIR = "./hyenadna_v9.9_checkpoints_32k"
    MODEL_NAME = "LongSafari/hyenadna-small-32k-seqlen-hf"
    
    # --- UPDATED FOR 32K CONTEXT ---
    MAX_LEN = 32768  # Full 32k Context
    BATCH_SIZE = 8   
    
    EPOCHS = 3
    LEARNING_RATE = 1e-5
    VAL_SPLIT = 0.2
    SEED = 42
    
    CONFIDENCE_THRESHOLD = 0.90 

if not os.path.exists(Config.OUTPUT_DIR):
    os.makedirs(Config.OUTPUT_DIR)

# --- REPRODUCIBILITY ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(Config.SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Setup Complete. Device: {device}")

if torch.cuda.is_available():
    print(f"   GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# ==============================================================================
# CELL 2: CLASS DEFINITIONS (DataPrep, Dataset, Model) - FULL & CORRECTED
# ==============================================================================
import random
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import AutoModel

# --- 1. DATA PREPARATOR (Unchanged) ---
class DataPreparator:
    COLUMNS = ["Hgene","Hchr","Hbp","Hstrand","Tgene","Tchr","Tbp","Tstrand","5'-gene sequence (10Kb)","3'-gene sequence (10Kb)"]

    def __init__(self, config):
        self.cfg = config
        self.trans_table = str.maketrans("ATCGN", "TAGCN")

    def _trim_artifacts(self, sequence: str) -> str:
        if not isinstance(sequence, str): return ""
        return sequence.upper().strip()

    def _get_reverse_complement(self, sequence: str) -> str:
        return sequence.upper().translate(self.trans_table)[::-1]

    def _generate_random_dna(self, length: int) -> str:
        if length <= 0: return ""
        return "".join(random.choices("ACGT", k=length))

    def _apply_jitter(self, seq_5p: str, seq_3p: str) -> str:
        """
        Combines sequences and pads with RANDOM DNA (ACGT) to reach 32k.
        Implements rigorous Random Padding for Translation Invariance.
        """
        seq_5p = self._trim_artifacts(seq_5p)
        seq_3p = self._trim_artifacts(seq_3p)
        
        # Combine to form core fusion sequence
        core_seq = seq_5p + seq_3p
        core_len = len(core_seq)
        target_len = self.cfg.MAX_LEN
        
        # Case A: Sequence is shorter than 32k -> Random Padding
        if core_len < target_len:
            needed = target_len - core_len
            
            # Uniform Random Placement
            pad_left = random.randint(0, needed)
            pad_right = needed - pad_left
            
            # Generate random ACGT noise (Signal Camouflage)
            left_seq = self._generate_random_dna(pad_left)
            right_seq = self._generate_random_dna(pad_right)
            
            return left_seq + core_seq + right_seq

        # Case B: Sequence is larger -> Random Crop
        else:
            # Pick any valid window from the long sequence
            max_start = core_len - target_len
            start = random.randint(0, max_start)
            return core_seq[start : start + target_len]

    def _load_fasta(self, path: str) -> list:
        if not os.path.exists(path): return []
        seqs, curr = [], []
        with open(path, 'r') as f:
            for line in f:
                line = line.strip()
                if line.startswith('>'):
                    if curr: seqs.append(self._trim_artifacts("".join(curr)))
                    curr = []
                else: curr.append(line)
            if curr: seqs.append(self._trim_artifacts("".join(curr)))
        return seqs

# --- 2. DATASET CLASS (Restored) ---
class DNABreakpointDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_len):
        self.sequences = sequences
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self): return len(self.sequences)
    
    def __getitem__(self, idx):
        seq = str(self.sequences[idx])
        label = self.labels[idx]
        
        # Tokenization for HyenaDNA (Character level handled by tokenizer)
        enc = self.tokenizer(
            seq, 
            truncation=True, 
            max_length=self.max_len, 
            padding='max_length', 
            return_tensors='pt'
        )
        
        input_ids = enc['input_ids'].squeeze(0)
        
        # Manual attention mask creation if needed
        if 'attention_mask' in enc:
            mask = enc['attention_mask'].squeeze(0)
        else:
            mask = (input_ids != (self.tokenizer.pad_token_id or 0)).long()
            
        return {'input_ids': input_ids, 'attention_mask': mask, 'labels': torch.tensor(label, dtype=torch.long)}

# --- 3. MODEL CLASS (Updated for Width Stats) ---
class HyenaDNAClassifier(nn.Module):
    def __init__(self, model_name, num_labels=2):
        super().__init__()
        # Load base model
        self.hyena = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        
        # Classification Head & Attention Pooling
        self.head = nn.Linear(self.hyena.config.d_model, 1) 
        self.clf = nn.Linear(self.hyena.config.d_model, num_labels)
        
    def forward(self, input_ids, attention_mask=None, labels=None):
        # 1. Get Embeddings
        out = self.hyena(input_ids).last_hidden_state
        
        # 2. Attention/Pooling Mechanism
        scores = self.head(out)
        if attention_mask is not None: 
            scores = scores.masked_fill(attention_mask.unsqueeze(-1) == 0, float('-inf'))
        
        # Capture the attention probabilities (The "Confidence Distribution" across the sequence)
        attn_probs = torch.softmax(scores, dim=1)
        
        # 3. Weighted Pooling
        pooled = torch.sum(out * attn_probs, dim=1)
        
        # 4. Final Classification
        logits = self.clf(pooled)
        
        # 5. Loss Calculation
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
            
        # 6. Breakpoint Estimation (Heuristic: Max Attention)
        bp_index = torch.argmax(attn_probs, dim=1).squeeze(-1)
        
        # --- RETURN DICT UPDATED WITH 'attention_probs' ---
        return {
            'loss': loss, 
            'logits': logits, 
            'breakpoint_index': bp_index,
            'attention_probs': attn_probs.squeeze(-1) # Shape: (Batch, SeqLen)
        }

In [ ]:
# ==============================================================================
# CELL 3: METRICS & VISUALIZATION LOGIC
# ==============================================================================
def compute_metrics(y_true, y_probs):
    y_pred = np.argmax(y_probs, axis=1)
    
    # --- Standard Metrics ---
    acc = accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    
    # --- AUC Metrics ---
    try: 
        auroc = roc_auc_score(y_true, y_probs[:, 1])
    except ValueError: 
        auroc = 0.0
        
    return {
        "acc": acc, "mcc": mcc, "f1": f1, "prec": prec, "rec": rec, "auroc": auroc
    }

def plot_comprehensive_results(y_true, y_probs, predicted_bps, title_prefix=""):
    y_pred = np.argmax(y_probs, axis=1)
    y_scores = y_probs[:, 1]
    
    plt.figure(figsize=(14, 10)) 
    
    # A. Confusion Matrix
    plt.subplot(2, 2, 1)
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
    plt.title(f'{title_prefix} Confusion Matrix')

    # B. ROC Curve
    plt.subplot(2, 2, 2)
    fpr, tpr, _ = roc_curve(y_true, y_scores)
    plt.plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_true, y_scores):.4f}', color='purple', lw=2)
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.legend()
    plt.title(f'{title_prefix} ROC Curve')

    # C. Probability Distribution
    plt.subplot(2, 2, 3)
    neg_scores = y_scores[y_true == 0]
    pos_scores = y_scores[y_true == 1]
    sns.histplot(neg_scores, color='red', alpha=0.5, label='True Neg', bins=30, kde=True)
    sns.histplot(pos_scores, color='green', alpha=0.5, label='True Pos', bins=30, kde=True)
    plt.axvline(0.5, color='black', linestyle='--')
    plt.legend()
    plt.title(f'{title_prefix} Probability Dist.')

    # D. Genomic Index
    plt.subplot(2, 2, 4)
    high_conf = np.where((y_pred == 1) & (y_scores > Config.CONFIDENCE_THRESHOLD))[0]
    if len(high_conf) > 0:
        sns.histplot(predicted_bps[high_conf], kde=True, bins=50, color='blue', label='Predicted')
        plt.title(f'{title_prefix} Breakpoints (Conf > {Config.CONFIDENCE_THRESHOLD})')
        plt.xlabel('Genomic Index')
    else:
        plt.text(0.5, 0.5, "No High-Confidence Fusions", ha='center')
        plt.title('Breakpoint Locations')

    plt.tight_layout()
    plt.show()

In [ ]:
# ==============================================================================
# CELL 4: TRAINING & VALIDATION LOOPS (CORRECTED)
# ==============================================================================

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    all_probs, all_labels = [], []
    
    for batch in tqdm(loader, desc="Training"):
        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        
        optimizer.zero_grad()
        outputs = model(ids, attention_mask=mask, labels=lbls)
        loss = outputs['loss']
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        probs = torch.softmax(outputs['logits'], dim=1)
        
        all_probs.extend(probs.detach().cpu().numpy())
        all_labels.extend(lbls.detach().cpu().numpy())
        
        # Memory Cleanup
        del ids, mask, lbls, outputs, loss
    
    if torch.cuda.is_available(): torch.cuda.empty_cache()
        
    metrics = compute_metrics(np.array(all_labels), np.array(all_probs))
    metrics['loss'] = total_loss / len(loader)
    return metrics

def validate_epoch(model, loader, criterion, device, desc="Validating"):
    model.eval()
    all_probs, all_labels, all_bps = [], [], []
    all_starts, all_ends, all_widths = [], [], [] 
    val_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(loader, desc=desc):
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            lbls = batch['labels'].to(device)
            
            outputs = model(ids, attention_mask=mask, labels=lbls)
            val_loss += outputs['loss'].item()
            
            probs = torch.softmax(outputs['logits'], dim=1)
            bps = outputs['breakpoint_index']
            
            # --- REAL WIDTH CALCULATION ---
            # Extract attention maps: (Batch, SeqLen)
            attn_maps = outputs['attention_probs'] 
            
            # Calculate FWHM (Full Width at Half Maximum)
            batch_starts, batch_ends, batch_widths = [], [], []
            
            for i in range(len(attn_maps)):
                # Get single sequence attention
                a_map = attn_maps[i]
                peak_idx = bps[i].item()
                peak_val = a_map[peak_idx].item()
                
                # Threshold: 50% of the peak value
                threshold = peak_val * 0.5
                
                # Find all indices above threshold
                # (We treat the region as the continuous span around the peak)
                above_thresh = (a_map >= threshold).nonzero(as_tuple=True)[0]
                
                if len(above_thresh) > 0:
                    start = above_thresh.min().item()
                    end = above_thresh.max().item()
                    width = end - start
                else:
                    start, end, width = peak_idx, peak_idx, 0
                    
                batch_starts.append(start)
                batch_ends.append(end)
                batch_widths.append(width)
            
            all_starts.extend(batch_starts)
            all_ends.extend(batch_ends)
            all_widths.extend(batch_widths)
            # -------------------------------

            all_probs.extend(probs.detach().cpu().numpy())
            all_labels.extend(lbls.detach().cpu().numpy())
            all_bps.extend(bps.detach().cpu().numpy())
            
            del ids, mask, lbls, outputs
            
    metrics = compute_metrics(np.array(all_labels), np.array(all_probs))
    metrics['loss'] = val_loss / len(loader)
    
    return metrics, np.array(all_labels), np.array(all_probs), np.array(all_bps), (np.array(all_starts), np.array(all_ends), np.array(all_widths))

In [ ]:
# ==============================================================================
# CELL 5: LOAD DATA & PERFORM STRICT GENE-PAIR SPLIT (EXPERIMENT 1)
# ==============================================================================
import hashlib
import pandas as pd
from sklearn.model_selection import train_test_split

print("\n--- 1. LOADING ALL DATA ---")
preparator = DataPreparator(Config)

# Track lengths for statistics
all_pos_lengths = []
all_neg_lengths = []

# =======================================================
# A. LOAD POSITIVES (CSV + FASTA)
# =======================================================
pos_dfs = []

# 1. Load Positive CSV
if os.path.exists(Config.POSITIVE_CSV):
    try:
        print(f"  > Loading Positive CSV: {Config.POSITIVE_CSV}")
        csv_df = pd.read_csv(Config.POSITIVE_CSV, header=None, names=DataPreparator.COLUMNS, sep='\t')
        
        # Stats (Raw Lengths)
        raw_lens = (
            csv_df["5'-gene sequence (10Kb)"].astype(str).str.strip().str.len() + 
            csv_df["3'-gene sequence (10Kb)"].astype(str).str.strip().str.len()
        )
        all_pos_lengths.extend(raw_lens.tolist())

        # Process
        csv_df['sequence'] = csv_df.apply(
            lambda r: preparator._apply_jitter(r["5'-gene sequence (10Kb)"], r["3'-gene sequence (10Kb)"]), 
            axis=1
        )
        csv_df['label'] = 1
        
        # Metadata for splitting (Fusion Name)
        csv_df['fusion_name'] = csv_df['Hgene'] + "--" + csv_df['Tgene']
        
        pos_dfs.append(csv_df)
        print(f"    - Loaded {len(csv_df)} samples from CSV.")
    except Exception as e:
        print(f"    ⚠️ Error reading Positive CSV: {e}")

# 2. Extra FASTA Positives
if hasattr(Config, 'EXTRA_POS_FASTA') and Config.EXTRA_POS_FASTA:
    print("  > Loading Extra FASTA Positives...")
    extra_seqs = []
    for p in Config.EXTRA_POS_FASTA.split(','):
        p = p.strip()
        if p and os.path.exists(p):
            loaded_raw = preparator._load_fasta(p)
            all_pos_lengths.extend([len(s) for s in loaded_raw])
            
            processed = [preparator._apply_jitter(s, "") for s in loaded_raw]
            extra_seqs.extend(processed)
            print(f"    - Loaded {len(processed)} from {os.path.basename(p)}")
            
    if extra_seqs:
        # FASTA files don't have gene names, so we leave 'fusion_name' null (will use sequence hash)
        pos_dfs.append(pd.DataFrame({'sequence': extra_seqs, 'label': 1}))

# Combine Positives
if pos_dfs:
    pos_df = pd.concat(pos_dfs, ignore_index=True)
    pos_df.drop_duplicates(subset=['sequence'], inplace=True)
else:
    pos_df = pd.DataFrame(columns=['sequence', 'label'])

# =======================================================
# B. LOAD NEGATIVES (CSV + FASTA)
# =======================================================
neg_dfs = []

# 1. Negative CSV
if os.path.exists(Config.NEGATIVE_CSV):
    try:
        print(f"  > Loading Negative CSV: {Config.NEGATIVE_CSV}")
        neg_csv = pd.read_csv(Config.NEGATIVE_CSV, header=None, names=DataPreparator.COLUMNS, sep='\t')
        
        # Stats
        raw_lens_neg = (
            neg_csv["5'-gene sequence (10Kb)"].astype(str).str.strip().str.len() + 
            neg_csv["3'-gene sequence (10Kb)"].astype(str).str.strip().str.len()
        )
        all_neg_lengths.extend(raw_lens_neg.tolist())

        # Process
        neg_csv['sequence'] = neg_csv.apply(
            lambda r: preparator._apply_jitter(r["5'-gene sequence (10Kb)"], r["3'-gene sequence (10Kb)"]), 
            axis=1
        )
        neg_csv['label'] = 0
        neg_csv['gene_name'] = neg_csv['Hgene'] # Negatives usually single gene
        
        # Augment (Reverse Complement) - Keep metadata same to force them into same split
        print("    > Augmenting CSV Negatives (Reverse Complement)...")
        aug_csv = neg_csv.copy()
        aug_csv['sequence'] = aug_csv['sequence'].apply(preparator._get_reverse_complement)
        neg_csv = pd.concat([neg_csv, aug_csv], ignore_index=True)
        
        neg_dfs.append(neg_csv)
    except Exception as e:
        print(f"    ⚠️ Error reading Negative CSV: {e}")

# 2. FASTA Negatives
neg_fasta_sources = [Config.NEG_FASTA_CANONICAL, Config.NEG_FASTA_SYNTHETIC]
for p in neg_fasta_sources:
    if os.path.exists(p):
        print(f"  > Loading Negative FASTA: {os.path.basename(p)}")
        raw_negs = preparator._load_fasta(p)
        all_neg_lengths.extend([len(s) for s in raw_negs])
        processed_negs = [preparator._apply_jitter(s, "") for s in raw_negs]
        neg_dfs.append(pd.DataFrame({'sequence': processed_negs, 'label': 0}))

# Combine Negatives
if neg_dfs:
    neg_df = pd.concat(neg_dfs, ignore_index=True)
    neg_df.drop_duplicates(subset=['sequence'], inplace=True)
else:
    neg_df = pd.DataFrame(columns=['sequence', 'label'])

# =======================================================
# C. STRICT GENE-PAIR HOLDOUT SPLIT
# =======================================================
print("\n--- PERFORMING STRICT BIOLOGICAL SPLIT ---")
full_df = pd.concat([pos_df, neg_df], ignore_index=True)

def get_biological_group_id(row):
    """Generates a Group ID to ensure biological variants stay together."""
    # 1. Use explicit metadata if available
    if row['label'] == 1 and 'fusion_name' in row and pd.notna(row['fusion_name']):
        return f"POS_{row['fusion_name']}"
    elif row['label'] == 0 and 'gene_name' in row and pd.notna(row['gene_name']):
        return f"NEG_{row['gene_name']}"
    
    # 2. Fallback: Hash the sequence content (ignoring padding if possible, or just raw)
    # This ensures duplicates or near-duplicates get grouped if metadata is missing
    s = str(row['sequence']).upper().strip()
    return hashlib.md5(s.encode()).hexdigest()

# Apply Grouping
full_df['split_group_id'] = full_df.apply(get_biological_group_id, axis=1)
unique_groups = full_df['split_group_id'].unique()

print(f"✅ Total Unique Samples: {len(full_df)}")
print(f"🧬 Unique Biological Groups: {len(unique_groups)} (Splitting based on this)")

# Split GROUPS, not samples
train_groups, temp_groups = train_test_split(unique_groups, test_size=0.30, random_state=Config.SEED)
val_groups, test_groups = train_test_split(temp_groups, test_size=0.50, random_state=Config.SEED)

# Assign rows
train_df = full_df[full_df['split_group_id'].isin(train_groups)].copy()
val_df   = full_df[full_df['split_group_id'].isin(val_groups)].copy()
test_df  = full_df[full_df['split_group_id'].isin(test_groups)].copy()

print(f"\n--- FINAL SPLIT SIZES ---")
print(f"TRAIN: {len(train_df)} samples")
print(f"VAL:   {len(val_df)} samples")
print(f"TEST:  {len(test_df)} samples")

# Leakage Verification
overlap = set(train_df['split_group_id']).intersection(set(test_df['split_group_id']))
if not overlap:
    print("✅ LEAKAGE CHECK PASSED: No biological events shared between Train and Test.")
else:
    print(f"❌ LEAKAGE DETECTED: {len(overlap)} shared groups!")

In [ ]:
# ==============================================================================
# CELL 6: INITIALIZE & TRAIN MODEL (CORRECTED)
# ==============================================================================
import torch
import torch.nn as nn
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup, AutoTokenizer
from torch.utils.data import DataLoader
from safetensors.torch import save_file 

# 1. Ensure labels are available
train_seqs_list = train_df['sequence'].tolist()
train_lbls_list = train_df['label'].tolist()
val_seqs_list = val_df['sequence'].tolist()
val_lbls_list = val_df['label'].tolist()

# 2. Print Stats
n_train_pos = sum(train_lbls_list)
n_train_neg = len(train_lbls_list) - n_train_pos
n_val_pos = sum(val_lbls_list)
n_val_neg = len(val_lbls_list) - n_val_pos

print(f"--- DATASET STATISTICS ---")
print(f"Training Set:   {len(train_seqs_list)} samples")
print(f"  ├── Positives: {n_train_pos} ({n_train_pos/len(train_seqs_list):.1%})")
print(f"  └── Negatives: {n_train_neg} ({n_train_neg/len(train_seqs_list):.1%})")
print(f"Validation Set: {len(val_seqs_list)} samples")
print(f"  ├── Positives: {n_val_pos} ({n_val_pos/len(val_seqs_list):.1%})")
print(f"  └── Negatives: {n_val_neg} ({n_val_neg/len(val_seqs_list):.1%})")

# 3. Initialize Tokenizer & Model
print(f"\nInit Tokenizer & Model: {Config.MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = HyenaDNAClassifier(Config.MODEL_NAME)
model.to(device)

# 4. Create Datasets & Dataloaders
train_dataset = DNABreakpointDataset(train_seqs_list, train_lbls_list, tokenizer, Config.MAX_LEN)
val_dataset = DNABreakpointDataset(val_seqs_list, val_lbls_list, tokenizer, Config.MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)

# --- NEW: Create Test Loader (Critical for Cell 7) ---
if 'test_df' in globals():
    test_seqs_list = test_df['sequence'].tolist()
    test_lbls_list = test_df['label'].tolist()
    test_dataset = DNABreakpointDataset(test_seqs_list, test_lbls_list, tokenizer, Config.MAX_LEN)
    test_loader = DataLoader(test_dataset, batch_size=Config.BATCH_SIZE, shuffle=False)
else:
    print("⚠️ Warning: 'test_df' not found. Test loader will not be created.")
# -----------------------------------------------------

# 5. Optimizer & Scheduler
criterion = nn.CrossEntropyLoss() # <--- FIXED: Defined criterion
optimizer = AdamW(model.parameters(), lr=Config.LEARNING_RATE)
total_steps = len(train_loader) * Config.EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=total_steps)

# 6. Training Loop
print(f"\n Training Start ({Config.EPOCHS} Epochs)...")
best_val_auc = 0.0

for epoch in range(Config.EPOCHS):
    print(f"\n--- Epoch {epoch + 1} ---")
    
    # Train
    train_metrics = train_epoch(model, train_loader, optimizer, scheduler, device)
    print(f"TRAIN >> Loss: {train_metrics['loss']:.4f} | Acc: {train_metrics['acc']:.4f} | F1: {train_metrics['f1']:.4f} | AUC: {train_metrics['auroc']:.4f}")
    
    # Validate
    # FIXED: Unpack the tuple correctly
    val_metrics, val_true, val_probs, val_bps, (val_starts, val_ends, val_widths) = validate_epoch(model, val_loader, criterion, device)
    print(f"VAL   >> Loss: {val_metrics['loss']:.4f} | Acc: {val_metrics['acc']:.4f} | F1: {val_metrics['f1']:.4f} | AUC: {val_metrics['auroc']:.4f}")
    
    # Save Best Model
    if val_metrics['auroc'] > best_val_auc:
        best_val_auc = val_metrics['auroc']
        save_path = os.path.join(Config.OUTPUT_DIR, "best_model.safetensors")
        
        # Clone tensors to break shared memory (Safetensors fix)
        state_dict_to_save = {k: v.clone() for k, v in model.state_dict().items()}
        save_file(state_dict_to_save, save_path)
        
        print(f"  ★ New Best Model Saved (AUC: {best_val_auc:.4f})")

In [ ]:
# ==============================================================================
# CELL 7: BLIND TEST SET EVALUATION (Full Feature Set & Experiment 2 Ready)
# ==============================================================================
import os
import torch
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
from torch.utils.data import DataLoader
from datetime import datetime
from safetensors.torch import load_file

print(f"{'='*40}")
print(f"PHASE 9: BLIND TEST SET EVALUATION")
print(f"{'='*40}")

# --- 1. ROBUST MODEL INITIALIZATION ---
if 'model' not in locals():
    print("🔄 Initializing Model Architecture...")
    try:
        model = HyenaDNAClassifier(Config.MODEL_NAME)
    except NameError:
        # Fallback using string literal if Config isn't defined
        model = HyenaDNAClassifier('hyenadna-medium-32k-seqlen') 

model.to(device)

# --- 2. LOAD BEST WEIGHTS ---
best_model_path = os.path.join(Config.OUTPUT_DIR, "best_model.safetensors")

# Fallback to .pt if .safetensors doesn't exist
if not os.path.exists(best_model_path):
    pt_path = os.path.join(Config.OUTPUT_DIR, "best_model.pt")
    if os.path.exists(pt_path):
        best_model_path = pt_path

if os.path.exists(best_model_path):
    print(f"📂 Loading Best Model: {best_model_path}")
    try:
        if best_model_path.endswith('.safetensors'):
            state_dict = load_file(best_model_path)
        else:
            state_dict = torch.load(best_model_path, map_location=device)
            
        model.load_state_dict(state_dict)
        print("✅ Weights loaded successfully.")
    except Exception as e:
        print(f"⚠️ Warning: Failed to load weights ({e}). Using current model state.")
else:
    print("⚠️ Warning: Best model file not found, using current model state.")

model.eval()

# --- 3. RUN INFERENCE ---
if 'criterion' not in locals(): criterion = torch.nn.CrossEntropyLoss()

if 'test_loader' in locals():
    # Validate and unpack all geometric data
    test_metrics, t_true, t_probs, t_bps, (t_starts, t_ends, t_widths) = validate_epoch(
        model, test_loader, criterion, device, desc="Testing"
    )

    # 4. PRINT STANDARD METRICS
    print(f"\n🏆 FINAL TEST RESULTS:")
    print(f"   Accuracy:  {test_metrics['acc']:.4f}")
    print(f"   AUC:       {test_metrics['auroc']:.4f}")
    print(f"   Precision: {test_metrics['prec']:.4f}")
    print(f"   Recall:    {test_metrics['rec']:.4f}")
    print(f"   F1 Score:  {test_metrics['f1']:.4f}")
    print(f"   MCC:       {test_metrics['mcc']:.4f}")

    # 5. EXPORT DETAILED CSV
    print(f"\n💾 EXPORTING DETAILED RESULTS...")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_filename = f"blind_test_predictions_detailed_{timestamp}.csv"

    # Create Results DataFrame
    if 'test_df' in locals() and len(test_df) == len(t_true):
        results_df = test_df.copy()
    else:
        results_df = pd.DataFrame() 

    # Add Predictions
    pred_indices = np.argmax(t_probs, axis=1)
    results_df['True_Label'] = t_true
    results_df['Predicted_Label'] = pred_indices
    results_df['Fusion_Prob'] = t_probs[:, 1]

    # Add Geometric Metadata (Crucial for later analysis)
    results_df['BP_Peak_Index'] = t_bps
    results_df['BP_Region_Start'] = t_starts.astype(int)
    results_df['BP_Region_End'] = t_ends.astype(int)
    results_df['BP_Region_Width'] = t_widths.astype(int)

    # Add Result Type (TP, FP, etc.)
    results_df['Result_Type'] = results_df.apply(
        lambda x: 'TP' if (x['True_Label'] == 1 and x['Predicted_Label'] == 1) else
                  ('TN' if (x['True_Label'] == 0 and x['Predicted_Label'] == 0) else
                  ('FP' if (x['True_Label'] == 0 and x['Predicted_Label'] == 1) else 'FN')),
        axis=1
    )

    # Save timestamped copy AND generic copy
    save_path = os.path.join(Config.OUTPUT_DIR, csv_filename)
    generic_path = os.path.join(Config.OUTPUT_DIR, "final_test_results_detailed.csv")
    
    results_df.to_csv(save_path, index=False)
    results_df.to_csv(generic_path, index=False)
    
    print(f"✅ Predictions saved to: {save_path}")
    print(f"   (And overwritten to: {generic_path})")

    # 6. FULL VISUALIZATION SUITE
    def plot_test_results_detailed(y_true, y_probs, predicted_bps):
        y_pred = np.argmax(y_probs, axis=1)
        y_scores = y_probs[:, 1] 
        
        plt.figure(figsize=(14, 10))
        
        # A. Confusion Matrix
        plt.subplot(2, 2, 1)
        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=['Non-Fusion', 'Fusion'], 
                    yticklabels=['Non-Fusion', 'Fusion'])
        plt.title('A. Confusion Matrix')

        # B. ROC Curve
        plt.subplot(2, 2, 2)
        fpr, tpr, _ = roc_curve(y_true, y_scores)
        plt.plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_true, y_scores):.4f}', color='purple', lw=2)
        plt.plot([0, 1], [0, 1], 'k--', lw=1)
        plt.xlabel('False Positive Rate (FPR)')
        plt.ylabel('True Positive Rate (TPR)')
        plt.legend()
        plt.title('B. ROC Curve')

        # C. Breakpoint Probability Distribution
        plt.subplot(2, 2, 3)
        neg_scores = y_scores[y_true == 0]
        pos_scores = y_scores[y_true == 1]
        sns.histplot(neg_scores, color='red', alpha=0.5, label='True Neg', bins=30, kde=True)
        sns.histplot(pos_scores, color='green', alpha=0.5, label='True Pos', bins=30, kde=True)
        plt.legend()
        plt.title('C. Confidence Distribution')

        # D. Breakpoint Location Histogram (High Confidence Only)
        plt.subplot(2, 2, 4)
        threshold = getattr(Config, 'CONFIDENCE_THRESHOLD', 0.9)
        high_conf_indices = np.where((y_pred == 1) & (y_scores > threshold))[0]
        if len(high_conf_indices) > 0:
            valid_bps = predicted_bps[high_conf_indices]
            sns.histplot(valid_bps, kde=True, bins=50, color='blue', label='Pred Breakpoints')
            plt.xlabel('Genomic Index (0-32k)')
            plt.title(f'D. Breakpoint Locations (Conf > {threshold})')
        else:
            plt.text(0.5, 0.5, "No High-Confidence Fusions", ha='center')

        plt.tight_layout()
        plt.show()

    plot_test_results_detailed(t_true, t_probs, t_bps)
    
    # --- CRITICAL FIX FOR EXPERIMENT 2 ---
    # Alias the variables so Cell 10 finds them immediately
    test_true = t_true
    test_probs = t_probs
    print("\n✅ Data variables 'test_true' and 'test_probs' are ready for Experiment 2.")

else:
    print("❌ Critical Error: 'test_loader' is not defined. Please run the Data Loading cell.")

In [ ]:
# ==============================================================================
# CELL 11: DETAILED ERROR & GEOMETRIC ANALYSIS (Fixed Column Names)
# ==============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# 1. LOAD DATA
filename = "final_test_results_detailed.csv"
# Ensure we look in the correct output directory if defined, else current dir
load_dir = Config.OUTPUT_DIR if 'Config' in locals() else ""
file_path = os.path.join(load_dir, filename)

if not os.path.exists(file_path):
    # Fallback to local directory if not found in output dir
    if os.path.exists(filename):
        file_path = filename
    else:
        print(f"❌ Error: Could not find {filename}")
        print("   Please run Cell 7 (Test Evaluation) first to generate it.")
        file_path = None

if file_path:
    df = pd.read_csv(file_path)
    print(f"✅ Loaded {len(df)} predictions from {file_path}")

    # 2. CALCULATE RESULT TYPES
    # We use 'Predicted_Label' because that matches Cell 7's output
    # TP: True=1, Pred=1 | FP: True=0, Pred=1 | FN: True=1, Pred=0 | TN: True=0, Pred=0
    
    conditions = [
        (df['True_Label'] == 1) & (df['Predicted_Label'] == 1),
        (df['True_Label'] == 0) & (df['Predicted_Label'] == 1),
        (df['True_Label'] == 1) & (df['Predicted_Label'] == 0),
        (df['True_Label'] == 0) & (df['Predicted_Label'] == 0)
    ]
    choices = ['TP', 'FP', 'FN', 'TN']
    df['Result_Type'] = np.select(conditions, choices, default='ERR')
    
    # Save back annotated version
    save_path = os.path.join(load_dir, "final_test_results_annotated.csv")
    df.to_csv(save_path, index=False)

    # 3. PRINT STATISTICS
    counts = df['Result_Type'].value_counts()
    print("\n📊 CONFUSION MATRIX COUNTS:")
    print(f"   True Positives (TP): {counts.get('TP', 0)}")
    print(f"   False Positives (FP): {counts.get('FP', 0)}")
    print(f"   False Negatives (FN): {counts.get('FN', 0)}")
    print(f"   True Negatives (TN): {counts.get('TN', 0)}")

    # 4. ANALYZE BREAKPOINT WIDTHS (Only for TP)
    tp_df = df[df['Result_Type'] == 'TP']
    
    # Check if 'BP_Region_Width' exists (it should from Cell 7)
    width_col = 'BP_Region_Width' if 'BP_Region_Width' in df.columns else 'Attention_Width'
    
    if len(tp_df) > 0 and width_col in tp_df.columns:
        widths = tp_df[width_col]
        median_w = widths.median()
        mean_w = widths.mean()
        sharp_count = (widths < 500).sum()
        
        print(f"\n📏 BREAKPOINT LOCALIZATION STATS (TP Only):")
        print(f"   Median Width: {median_w:.1f} bp")
        print(f"   Mean Width:   {mean_w:.1f} bp")
        print(f"   Sharp Signals (<500bp): {sharp_count} / {len(tp_df)} ({sharp_count/len(tp_df)*100:.1f}%)")

        # 5. HISTOGRAM OF WIDTHS
        plt.figure(figsize=(10, 5))
        plt.hist(widths, bins=50, color='purple', alpha=0.7, range=(0, 5000))
        plt.axvline(median_w, color='black', linestyle='--', label=f'Median: {median_w:.0f}bp')
        plt.title("Distribution of Predicted Breakpoint Widths (TP)", fontsize=14)
        plt.xlabel("Width (bp)")
        plt.ylabel("Count")
        plt.legend()
        plt.grid(alpha=0.3)
        plt.show()
    else:
        print("\n⚠️ No True Positives found or Width column missing.")

    # 6. ANALYZE CONFIDENCE
    # Compare confidence (prob) of Correct vs Incorrect predictions
    correct_mask = (df['Result_Type'] == 'TP') | (df['Result_Type'] == 'TN')
    
    # Use 'Fusion_Prob' from Cell 7
    prob_col = 'Fusion_Prob' if 'Fusion_Prob' in df.columns else 'Prob_Fusion'
    
    if prob_col in df.columns:
        # We map prob to "Confidence Score" = abs(prob - 0.5) * 2
        # (0.5 = 0% confidence, 1.0 or 0.0 = 100% confidence)
        df['Confidence'] = (df[prob_col] - 0.5).abs() * 2
        
        avg_conf_correct = df[correct_mask]['Confidence'].mean()
        avg_conf_wrong = df[~correct_mask]['Confidence'].mean()
        
        print("\n🧠 MODEL CERTAINTY:")
        print(f"   Avg Confidence on Correct Preds: {avg_conf_correct:.4f}")
        print(f"   Avg Confidence on Errors:        {avg_conf_wrong:.4f}")
        print("   (Higher gap is better. Close to 1.0 means 'Sure', 0.0 means 'Guessing')")

In [ ]:
# ==============================================================================
# CELL 9: GEOMETRIC VISUAL VERIFICATION 
# ==============================================================================
import matplotlib.pyplot as plt
import torch
import numpy as np
from tqdm import tqdm

def find_first_sharp_samples(model, loader, device, max_width_bp=250):
    """
    Scans the test set and stops as soon as it finds ONE sample per category
    that has a width smaller than max_width_bp (e.g., 250bp).
    """
    model.eval()
    
    # Categories we want to fill
    targets = ["Skewed 5'", "Centered", "Skewed 3'"]
    
    # Store found samples here
    found_samples = {k: None for k in targets}
    
    print(f"🔎 Scanning for first samples with width < {max_width_bp}bp...")
    
    # We iterate until we find all 3 or exhaust the loader
    for batch in tqdm(loader, desc="Scanning Geometry"):
        
        # Check if we are done
        if all(v is not None for v in found_samples.values()):
            break

        ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        
        # Skip if no positives
        if (lbls == 1).sum() == 0: continue

        # Inference
        with torch.no_grad():
            outputs = model(ids, attention_mask=mask)
        
        pos_indices = (lbls == 1).nonzero(as_tuple=True)[0]
        
        for idx in pos_indices:
            # Check if we are done inside the batch loop
            if all(v is not None for v in found_samples.values()):
                break

            attn_map = outputs['attention_probs'][idx].squeeze().cpu().numpy()
            
            # --- GEOMETRY ---
            peak_idx = np.argmax(attn_map)
            peak_val = attn_map[peak_idx]
            
            threshold = peak_val * 0.5
            above_thresh = np.where(attn_map >= threshold)[0]
            
            if len(above_thresh) > 0:
                start_idx = above_thresh.min()
                end_idx = above_thresh.max()
                width = end_idx - start_idx
            else:
                width = 99999 # invalid
            
            # --- FILTER: MUST BE SHARP (< 250bp) AND REAL (> 5bp) ---
            if width <= 5 or width >= max_width_bp:
                continue 

            # --- CLASSIFY ---
            window_len = len(attn_map)
            global_rel_pos = peak_idx / window_len
            
            if global_rel_pos < 0.35: 
                category = "Skewed 5'"
            elif global_rel_pos > 0.65: 
                category = "Skewed 3'"
            else: 
                category = "Centered"
            
            # --- STORE IF EMPTY ---
            if found_samples[category] is None:
                found_samples[category] = {
                    'map': attn_map,
                    'peak': peak_idx,
                    'peak_val': peak_val,
                    'start': start_idx,
                    'end': end_idx,
                    'width': width,
                    'rel_pos': global_rel_pos
                }
                print(f"   ✅ Found {category:<10} | Width: {width}bp")

    # --- PLOTTING ---
    print("\n📊 Plotting Identified Samples...")
    fig, axes = plt.subplots(3, 1, figsize=(15, 12))
    plt.subplots_adjust(hspace=0.5)
    
    for i, category in enumerate(targets):
        ax = axes[i]
        data = found_samples[category]
        
        if data is None:
            ax.text(0.5, 0.5, f"No sample < {max_width_bp}bp found for {category}", ha='center', va='center')
            continue
            
        attn_map = data['map']
        peak_idx = data['peak']
        width = data['width']
        
        # Zoom Logic
        zoom = max(200, width * 3) 
        view_min = max(0, peak_idx - zoom)
        view_max = min(len(attn_map), peak_idx + zoom)
        
        x = np.arange(view_min, view_max)
        y = attn_map[view_min:view_max]
        
        # Plot
        ax.plot(x, y, color='#2ca02c', label='Attention Profile', linewidth=2)
        ax.fill_between(x, y, color='#2ca02c', alpha=0.1)
        
        # Markers
        ax.axvline(peak_idx, color='red', linestyle='--', alpha=0.8, label=f'Peak ({peak_idx})')
        ax.axvspan(data['start'], data['end'], color='blue', alpha=0.1, label=f'Width ({width}bp)')
        
        # Format
        ax.set_xlim(view_min, view_max)
        ax.set_ylim(0, max(1.0, data['peak_val'] * 1.15))
        
        ax.set_title(f"Type: {category} | Global Pos: {data['rel_pos']:.2f} | Width: {width}bp", 
                     fontsize=12, fontweight='bold')
        ax.legend(loc='upper right')
        ax.grid(True, alpha=0.2)
        ax.set_xlabel("Genomic Coordinate")
        ax.set_ylabel("Prob")

    plt.show()

# Execute
find_first_sharp_samples(model, test_loader, device, max_width_bp=250)

In [ ]:
# ==============================================================================
# CELL 10: EXPERIMENT 2 - CLINICAL REALITY PR CURVE (With Printouts)
# ==============================================================================
from sklearn.metrics import precision_recall_curve, auc
import matplotlib.pyplot as plt
import numpy as np

def plot_clinical_impact(y_true, y_probs, clinical_prevalence=0.01):
    """
    Simulates performance in a clinical setting where fusions are rare (e.g. 1%).
    """
    # 1. Standard (Balanced) Curve
    precision_bal, recall_bal, _ = precision_recall_curve(y_true, y_probs)
    auc_bal = auc(recall_bal, precision_bal)
    
    # 2. Clinical Adjustment
    # Weight negatives higher to simulate 1:100 ratio
    n_pos = np.sum(y_true == 1)
    n_neg = np.sum(y_true == 0)
    
    # Calculate weight factor
    # Target: Pos / (Pos + Neg_Weighted) = 0.01
    neg_weight = (n_pos / n_neg) * ((1 - clinical_prevalence) / clinical_prevalence)
    
    # Adjust Precision: TP / (TP + FP * weight)
    # Derived from balanced precision: FP = TP * (1/Prec_bal - 1)
    with np.errstate(divide='ignore', invalid='ignore'):
        fp_ratio = (1 / precision_bal) - 1
        precision_clin = 1 / (1 + fp_ratio * neg_weight)
    
    # Clean up NaNs
    precision_clin = np.nan_to_num(precision_clin, nan=0.0)
    auc_clin = auc(recall_bal, precision_clin)

    # 3. PRINT KEY METRICS (Added this section)
    print("-" * 40)
    print(f"📊 CLINICAL METRICS REPORT (Prevalence={clinical_prevalence*100}%)")
    print("-" * 40)
    print(f"1. Clinical AUPRC:      {auc_clin:.4f}")
    
    # Metric A: Precision at 20% Recall
    idx_20 = (np.abs(recall_bal - 0.20)).argmin()
    prec_at_20 = precision_clin[idx_20]
    print(f"2. Precision at 20% Recall: {prec_at_20:.4f}")
    
    # Metric B: Max Recall at 90% Precision
    high_conf_indices = np.where(precision_clin >= 0.90)[0]
    if len(high_conf_indices) > 0:
        rec_at_90 = recall_bal[high_conf_indices].max()
        print(f"3. Recall at 90% Precision: {rec_at_90:.4f}")
    else:
        print("3. Recall at 90% Precision: 0.0000")

    # Metric C: Max Recall at 50% Precision (Drop-off point)
    break_even_indices = np.where(precision_clin >= 0.50)[0]
    if len(break_even_indices) > 0:
        rec_at_50 = recall_bal[break_even_indices].max()
        print(f"4. Drop-off (Recall @ 50% Prec): {rec_at_50:.4f}")
    else:
        print("4. Drop-off (Recall @ 50% Prec): 0.0000")
    print("-" * 40)

    # 4. Plotting
    plt.figure(figsize=(10, 7))
    plt.plot(recall_bal, precision_bal, color='blue', lw=2, alpha=0.6, 
             label=f'Balanced Test Set (AUC={auc_bal:.3f})')
    plt.plot(recall_bal, precision_clin, color='red', lw=3, 
             label=f'Clinical Scenario (1% Prev) (AUC={auc_clin:.3f})')
    
    # Baselines
    plt.axhline(y=clinical_prevalence, color='red', linestyle=':', label='Clinical Baseline (1%)')
    plt.axhline(y=0.5, color='blue', linestyle=':', label='Balanced Baseline (50%)')
    
    plt.title("Experiment 2: Clinical Utility Analysis (PR Curve)", fontsize=14)
    plt.xlabel("Recall (Sensitivity)", fontsize=12)
    plt.ylabel("Precision (Positive Predictive Value)", fontsize=12)
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.xlim([0, 1.0])
    plt.ylim([0, 1.05])
    plt.show()

# Execute using variables from Cell 7
if 'test_true' in locals() and 'test_probs' in locals():
    plot_clinical_impact(test_true, test_probs[:, 1], clinical_prevalence=0.01)
else:
    print("⚠️ Error: Run Cell 7 first to generate test predictions.")